# Проверка соответствия товарного знака наименованию товара — подробная русскоязычная development-версия

**Задача:** проверить критерий «Значение атрибута „Товарный знак“ не соответствует наименованию товара».

Эта версия сохраняет подробную логику разработки алгоритма, но рядом с основными английскими техническими терминами добавлены русские пояснения.

Технические статусы внутри кода сохраняются без изменений:

- `MATCH` — **соответствие**;
- `LIKELY_MATCH` — **вероятное соответствие**;
- `MISMATCH` — **несоответствие / противоречие**;
- `REVIEW` — **требуется ручная проверка**;
- `UNCERTAIN` — **неопределённый случай при ручной разметке**.

Это сохраняет совместимость с API, логами и дальнейшей интеграцией.

## Что было очищено и сохранено

В этой версии:

- удалены ошибочные и устаревшие ячейки;
- убраны старые определения `check_trademark` и вспомогательных функций;
- оставлена только финальная логика нормализации, `quality checks` (**проверок качества данных**), `association matching` (**поиска статистических ассоциаций**) и `direct conflict` (**прямого конфликта брендов**);
- объединены проверенные связи ароматов с брендами;
- итоговые метрики отделены от промежуточных экспериментов;
- сохранён переход от алгоритмического `baseline` (**базового решения**) к ML.

## Словарь основных терминов

| Термин | Русское пояснение |
|---|---|
| `MATCH` | соответствие товарного знака наименованию |
| `LIKELY_MATCH` | вероятное соответствие |
| `MISMATCH` | несоответствие / противоречие |
| `REVIEW` | требуется ручная проверка |
| `UNCERTAIN` | неопределённый случай |
| `Baseline` | базовое решение |
| `Exact match` | точное совпадение |
| `Fuzzy matching` | нечёткое / приблизительное сравнение строк |
| `Trusted brand` | доверенный бренд |
| `Association` | статистическая связь / ассоциация |
| `Coverage` | покрытие автоматическими решениями |
| `Accuracy` | доля правильных автоматических решений |
| `Precision` | точность срабатываний класса |
| `Recall` | полнота |
| `F1-score` | F1-мера |
| `Support` | число примеров класса |
| `Train/test split` | разделение на обучающую и тестовую выборки |
| `Data leakage` | утечка данных |
| `Inference` | применение готового алгоритма к новым данным |
| `Production` | рабочая эксплуатация системы |

> В ML термины `Precision`, `Recall`, `F1-score` обычно сохраняют на английском, поэтому рядом указано русское значение.

In [1]:
from pathlib import Path
import re

import pandas as pd
from rapidfuzz import fuzz
from sklearn.metrics import precision_recall_fscore_support

In [2]:
# Русские подписи только для отображения.
# Технические значения статусов и методов алгоритма не меняются.

STATUS_RU = {
    "MATCH": "MATCH (соответствие)",
    "LIKELY_MATCH": "LIKELY_MATCH (вероятное соответствие)",
    "MISMATCH": "MISMATCH (несоответствие)",
    "REVIEW": "REVIEW (ручная проверка)",
    "UNCERTAIN": "UNCERTAIN (неопределённый случай)",
}

METHOD_RU = {
    "missing_data": "отсутствуют необходимые данные",
    "input_quality_review": "проверка качества входного текста",
    "no_brand_conflict_review": "бренд не указан, но в тексте найден бренд",
    "no_brand_review": "бренд не указан — ручная проверка",
    "invalid_brand_format_review": "некорректный формат товарного знака",
    "exact_match": "точное совпадение",
    "verified_relation_match": "проверенная связь аромат/линейка → бренд",
    "verified_relation_conflict": "проверенная связь указывает на другой бренд",
    "association_match": "статистическая ассоциация подтверждает бренд",
    "direct_brand_conflict": "прямое упоминание другого бренда",
    "fuzzy_match": "нечёткое совпадение",
    "trusted_brand_conflict": "конфликт с доверенным брендом",
    "insufficient_evidence": "недостаточно данных для автоматического решения",
}

def status_ru(status):
    return STATUS_RU.get(status, status)

def method_ru(method):
    return METHOD_RU.get(method, method)


In [3]:
DATA_PATH = Path("perfumery_brands.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError(
        "Файл perfumery_brands.csv не найден. "
        "Положите его в ту же папку, что и ноутбук, и запустите ячейки заново."
    )

REQUIRED_COLUMNS = ["brand", "brand_in_text", "text"]

## 1. Загрузка данных

Для работы нужен `perfumery_brands.csv` в той же папке.

Используются поля:

- `brand` — заявленный товарный знак;
- `brand_in_text` — бренд, найденный/указанный в исходных данных;
- `text` — наименование товара.

`brand_in_text` используется при построении статистических справочников, но не является обязательным входом `inference`-функции (**функции применения готового алгоритма**).

In [4]:
data = pd.read_csv(
    DATA_PATH,
    usecols=REQUIRED_COLUMNS
)

print("Размер датасета:", data.shape)
data.head()

Размер датасета: (1494163, 3)


,brand,text,brand_in_text
0,ЮДИФ,Туалетная вода <Юдиф> Вишневый цвет 42мл,ЮДИФ
1,ЮДИФ,Туалетная вода <Юдиф> Нежная сирень жен 42мл,ЮДИФ
2,SHAIK PLATINUM,Парфюмерная вода W 332 SHAIK Platınum Fruity 50ML,SHAIK
3,SHAIK PLATINUM,Парфюмерная вода MW 333 SHAIK Platınum Leather...,SHAIK
4,SHAIK PLATINUM,Парфюмерная вода W 382 SHAIK Platınum Floral G...,SHAIK


## 2. Краткий анализ данных

Проверяются пропуски и количество уникальных значений перед построением справочников.

Это диагностический этап: он помогает понять качество и структуру исходного датасета.

In [5]:
data[REQUIRED_COLUMNS].isna().sum()

brand               510
brand_in_text    448064
text               1323
dtype: int64

In [6]:
data["brand"].nunique(), data["brand_in_text"].nunique()

(49500, 8572)

## Нормализация текста

`Normalization` (**нормализация**) приводит строки к единому виду:

- нижний регистр;
- `ё → е`;
- удаление пунктуации;
- удаление лишних пробелов.

Дополнительная нормализация обрабатывает `mixed-script` (**смешение кириллицы и латиницы**) внутри одного слова, например визуально похожие `р/p`, `а/a`, `м/m`.

In [7]:
def normalize_text(text):
    if pd.isna(text):
        return None

    text = str(text).lower()
    text = text.replace("ё", "е")
    text = re.sub(r"[^\w\s]", " ", text)
    text = re.sub(r"\s+", " ", text)

    return text.strip()

In [8]:
confusable_map = str.maketrans({
    "а": "a",
    "е": "e",
    "о": "o",
    "р": "p",
    "с": "c",
    "у": "y",
    "х": "x",
    "к": "k",
    "м": "m",
    "т": "t",
    "в": "b",
    "н": "h",
})


def normalize_for_matching(text):
    text = normalize_text(text)

    if text is None:
        return None

    result = []

    for word in text.split():

        has_latin = bool(
            re.search(r"[a-z]", word)
        )

        has_cyrillic = bool(
            re.search(r"[а-яё]", word)
        )

        # Исправляем только слова,
        # где одновременно есть латиница и кириллица
        if has_latin and has_cyrillic:
            word = word.translate(confusable_map)

        result.append(word)

    return " ".join(result)

## Служебные правила и проверенные связи

Здесь задаются:

- специальные значения отсутствующего товарного знака;
- подтверждённые `aliases` (**варианты написания брендов**);
- вручную проверенные связи `аромат/линейка → бренд`.

Проверенные связи имеют приоритет над шумными статистическими ассоциациями.

In [9]:
no_brand_values = {
    "без товарного знака",
    "безтоварного знака",
    "без бренда",
    "бренд отсутствует",
    "товарный знак отсутствует",
    "отсутствует",
    "нет",
    "без знака",
    "без товарной марки",
    "не указан",
}

brand_aliases = {
    "byredo parfums": "byredo",
    "francis kurkdjian": "maison francis kurkdjian",
    "parfum francis kurkdjian": "maison francis kurkdjian",
}

verified_product_relations = {
    "domenico gusto": "christine lavoisier parfums",
    "oudi": "al haramain",
    "russian police clean hands": "sergio nero",
    "the only one": "dolce gabbana",
    "stronger with you intensely": "giorgio armani",
    "festival splashes": "mexx",
    "ma cherry": "adopt",
    "hyrax": "zoologist",
}

## 3. Доверенный справочник брендов

Из исторических пар `brand_in_text → brand` строится `trusted brand catalog` (**доверенный справочник брендов**).

В него попадают только устойчивые значения с высокой долей доминирующего бренда (`dominance ≥ 0.95`).

Цель — использовать только достаточно надёжные брендовые сигналы и снизить риск ложного `MISMATCH`.

In [10]:
brand_data = data[["brand", "brand_in_text"]].copy()

brand_data["brand_normalized"] = brand_data["brand"].apply(normalize_text)
brand_data["brand_in_text_normalized"] = brand_data["brand_in_text"].apply(normalize_text)

clean_pairs_data = brand_data.dropna(
    subset=["brand_normalized", "brand_in_text_normalized"]
).copy()

clean_pairs_data = clean_pairs_data[
    (~clean_pairs_data["brand_normalized"].isin(no_brand_values))
    & (~clean_pairs_data["brand_normalized"].str.isnumeric())
]

clean_pair_counts = (
    clean_pairs_data
    .groupby(["brand_in_text_normalized", "brand_normalized"])
    .size()
    .reset_index(name="count")
)

# Доверенный справочник прямых брендов
clean_dominant_brand = (
    clean_pair_counts
    .sort_values(["brand_in_text_normalized", "count"], ascending=[True, False])
    .drop_duplicates("brand_in_text_normalized")
    .rename(columns={"brand_normalized": "dominant_brand", "count": "dominant_count"})
)

clean_candidate_totals = (
    clean_pair_counts
    .groupby("brand_in_text_normalized")["count"]
    .sum()
    .reset_index(name="total")
)

clean_dominant_stats = clean_dominant_brand.merge(
    clean_candidate_totals,
    on="brand_in_text_normalized"
)
clean_dominant_stats["dominance"] = (
    clean_dominant_stats["dominant_count"] / clean_dominant_stats["total"]
)

trusted_brands = clean_dominant_stats[
    (clean_dominant_stats["total"] >= 20)
    & (clean_dominant_stats["dominance"] >= 0.95)
    & (
        clean_dominant_stats["brand_in_text_normalized"]
        == clean_dominant_stats["dominant_brand"]
    )
].copy()

trusted_brand_catalog = trusted_brands["brand_in_text_normalized"].tolist()

print("Доверенных брендов:", len(trusted_brand_catalog))

Доверенных брендов: 316


In [11]:
def find_trusted_brands(text):
    text = normalize_text(text)

    if text is None:
        return []

    text_with_spaces = f" {text} "
    found = []

    for brand in trusted_brand_catalog:
        if len(brand) < 4:
            continue

        brand_with_spaces = f" {brand} "

        if brand_with_spaces in text_with_spaces:
            found.append(brand)

    return sorted(found, key=len, reverse=True)

In [12]:
def brands_equivalent(brand_1, brand_2):
    brand_1 = normalize_text(brand_1)
    brand_2 = normalize_text(brand_2)

    if brand_1 is None or brand_2 is None:
        return False

    # Полное совпадение
    if brand_1 == brand_2:
        return True

    # Один вариант входит в другой
    if brand_1 in brand_2 or brand_2 in brand_1:
        return True

    # Небольшая разница в написании
    similarity = fuzz.ratio(brand_1, brand_2)

    if similarity >= 90:
        return True

    return False

In [13]:
def detect_trusted_mismatch(brand, text):
    detected_brands = find_trusted_brands(text)

    conflicts = []

    for detected_brand in detected_brands:
        if not brands_equivalent(brand, detected_brand):
            conflicts.append(detected_brand)

    if conflicts:
        return "MISMATCH", conflicts

    return "NO_CONFLICT_FOUND", []

## 4. Канонизация и устойчивые статистические ассоциации

`Canonicalization` (**канонизация**) объединяет разные варианты одного бренда.

После этого строятся статистические связи между фрагментом наименования и брендом.

Текущие условия `baseline` (**базового решения**):

- `top1_count ≥ 20`;
- `top_ratio ≥ 3`;
- `count_gap ≥ 20`;
- длина термина ≥ 4.

Более длинные и конкретные совпадения имеют приоритет над вложенными короткими значениями.

In [14]:
def canonicalize_brand(brand):
    brand = normalize_text(brand)

    if brand is None:
        return None

    return brand_aliases.get(brand, brand)

In [15]:
canonical_pairs = clean_pair_counts.copy()
canonical_pairs["canonical_brand"] = canonical_pairs["brand_normalized"].apply(canonicalize_brand)

canonical_pair_counts = (
    canonical_pairs
    .groupby(["brand_in_text_normalized", "canonical_brand"], as_index=False)["count"]
    .sum()
)

canonical_ranked = canonical_pair_counts.copy()
canonical_ranked["rank"] = (
    canonical_ranked
    .groupby("brand_in_text_normalized")["count"]
    .rank(method="first", ascending=False)
)

canonical_top1 = (
    canonical_ranked[canonical_ranked["rank"] == 1]
    [["brand_in_text_normalized", "canonical_brand", "count"]]
    .rename(columns={"canonical_brand": "top1_brand", "count": "top1_count"})
)

canonical_top2 = (
    canonical_ranked[canonical_ranked["rank"] == 2]
    [["brand_in_text_normalized", "canonical_brand", "count"]]
    .rename(columns={"canonical_brand": "top2_brand", "count": "top2_count"})
)

canonical_strength = canonical_top1.merge(
    canonical_top2,
    on="brand_in_text_normalized",
    how="left"
)

canonical_strength["top_ratio"] = (
    canonical_strength["top1_count"] / canonical_strength["top2_count"]
)
canonical_strength["count_gap"] = (
    canonical_strength["top1_count"] - canonical_strength["top2_count"]
)

strong_associations = canonical_strength[
    (canonical_strength["top1_count"] >= 20)
    & (canonical_strength["top_ratio"] >= 3)
    & (canonical_strength["count_gap"] >= 20)
    & (canonical_strength["brand_in_text_normalized"].str.len() >= 4)
].copy()

association_map = dict(
    zip(
        strong_associations["brand_in_text_normalized"],
        strong_associations["top1_brand"]
    )
)

print("Устойчивых статистических связей:", len(association_map))

Устойчивых статистических связей: 747


In [16]:
def find_associated_brands(text):
    normalized_text = normalize_for_matching(text)

    if normalized_text is None:
        return []

    matches = []

    for term, associated_brand in association_map.items():
        normalized_term = normalize_for_matching(term)

        if normalized_term is None:
            continue

        # Ищем отдельное слово или фразу,
        # а не часть другого слова
        pattern = rf"(?<!\w){re.escape(normalized_term)}(?!\w)"

        for match in re.finditer(pattern, normalized_text):
            matches.append(
                {
                    "term": term,
                    "brand": associated_brand,
                    "start": match.start(),
                    "end": match.end(),
                    "length": match.end() - match.start()
                }
            )

    # Сначала самые длинные и конкретные совпадения
    matches.sort(
        key=lambda item: item["length"],
        reverse=True
    )

    selected = []

    for candidate in matches:
        # Проверяем, не находится ли кандидат
        # полностью внутри уже выбранного более длинного совпадения
        is_nested = any(
            candidate["start"] >= existing["start"]
            and candidate["end"] <= existing["end"]
            for existing in selected
        )

        if not is_nested:
            selected.append(candidate)

    # Технические координаты наружу не возвращаем
    return [
        {
            "term": item["term"],
            "brand": item["brand"]
        }
        for item in selected
    ]

## 5. Проверенные связи и проверки качества данных

Здесь находятся:

1. `Verified relations` — **проверенные связи аромат/линейка → бренд**.
2. `Input quality checks` — **проверки качества входного текста**.
3. `Brand quality checks` — **проверка формата значения товарного знака**.

Подозрительные входы безопаснее отправлять в `REVIEW` (**ручную проверку**), а не автоматически объявлять `MISMATCH`.

In [17]:
def find_verified_product_relation(text):
    normalized_text = normalize_for_matching(text)

    if normalized_text is None:
        return None

    # Сначала проверяем более длинные названия
    for term in sorted(
        verified_product_relations,
        key=len,
        reverse=True
    ):
        normalized_term = normalize_for_matching(term)

        # Ищем термин только как отдельное слово или фразу,
        # а не как часть другого слова
        pattern = rf"(?<!\w){re.escape(normalized_term)}(?!\w)"

        if re.search(pattern, normalized_text):
            return {
                "term": term,
                "brand": verified_product_relations[term]
            }

    return None

In [18]:
def check_input_quality(text):
    if pd.isna(text):
        return {
            "is_valid": False,
            "reason": "Отсутствует текст товара"
        }

    text = str(text)

    text_length = len(text)

    barcode_like_count = len(
        re.findall(
            r"\b\d{12,14}\b",
            text
        )
    )

    if text_length > 500:
        return {
            "is_valid": False,
            "text_length": text_length,
            "barcode_like_count": barcode_like_count,
            "reason": "Аномально длинное наименование товара"
        }

    if barcode_like_count >= 3:
        return {
            "is_valid": False,
            "text_length": text_length,
            "barcode_like_count": barcode_like_count,
            "reason": (
                "В тексте обнаружено несколько числовых "
                "идентификаторов, возможна склейка товаров"
            )
        }

    return {
        "is_valid": True,
        "text_length": text_length,
        "barcode_like_count": barcode_like_count
    }

In [19]:
def check_brand_input_quality(brand):
    if pd.isna(brand):
        return {
            "is_valid": False,
            "reason": "Отсутствует значение товарного знака"
        }

    normalized_brand = normalize_for_matching(brand)

    if normalized_brand is None:
        return {
            "is_valid": False,
            "reason": "Отсутствует значение товарного знака"
        }

    # 1. Значение состоит только из цифр
    if normalized_brand.isdigit():
        return {
            "is_valid": False,
            "reason": (
                "Значение товарного знака состоит только из цифр "
                "и требует отдельной проверки корректности атрибута"
            )
        }

    # 2. Сомнительные служебные значения текущего датасета
    suspicious_brand_values = {
        "aroma"
    }

    if normalized_brand in suspicious_brand_values:
        return {
            "is_valid": False,
            "reason": (
                "Значение товарного знака является сомнительным "
                "служебным значением в текущем датасете"
            )
        }

    # 3. Значение похоже на наименование юридического лица
    legal_entity_patterns = [
        r"^(ооо|оао|зао|пао|ао)\b",
        r"^ип\b",
        r"^общество с ограниченной ответственностью\b",
        r"^(llc|ltd|inc)\b"
    ]

    for pattern in legal_entity_patterns:
        if re.search(pattern, normalized_brand):
            return {
                "is_valid": False,
                "reason": (
                    "Значение похоже на наименование юридического "
                    "лица или производителя и требует отдельной проверки"
                )
            }

    return {
        "is_valid": True
    }

In [20]:
def find_direct_brand_conflict(brand, text):
    declared_brand = normalize_for_matching(
        canonicalize_brand(brand)
    )

    associations = find_associated_brands(text)

    for association in associations:
        term = normalize_for_matching(
            association["term"]
        )

        associated_brand = normalize_for_matching(
            canonicalize_brand(
                association["brand"]
            )
        )

        # Сильный случай:
        # найденный термин сам является названием бренда
        if term == associated_brand:

            # Если найденный бренд отличается от заявленного
            if declared_brand != associated_brand:
                return {
                    "brand": associated_brand,
                    "term": association["term"]
                }

    return None

## 6. Финальная функция проверки

`check_trademark(brand, text)` — итоговая `inference`-функция (**применение готового алгоритма к новым данным**).

Порядок принятия решения:

1. отсутствующие данные;
2. качество входного текста;
3. специальные значения отсутствующего бренда;
4. качество значения `brand`;
5. `exact match` — точное совпадение;
6. `verified relations` — проверенные связи;
7. `statistical associations` — статистические ассоциации;
8. `direct brand conflict` — прямой конфликт брендов;
9. `fuzzy matching` — нечёткое сравнение;
10. `trusted brand conflict` — конфликт с доверенным брендом;
11. недостаточно доказательств → `REVIEW` (**ручная проверка**).

In [21]:
def check_trademark(brand, text):
    # Нормализация
    normalized_brand = normalize_for_matching(brand)
    normalized_text = normalize_for_matching(text)

    # 1. Проверка отсутствующих данных
    if normalized_brand is None or normalized_text is None:
        return {
            "status": "REVIEW",
            "declared_brand": brand,
            "method": "missing_data",
            "reason": "Отсутствуют необходимые данные"
        }

    # 2. Проверка качества входного текста
    quality_result = check_input_quality(text)

    if not quality_result["is_valid"]:
        return {
            "status": "REVIEW",
            "declared_brand": brand,
            "method": "input_quality_review",
            "text_length": quality_result.get("text_length"),
            "barcode_like_count": quality_result.get(
                "barcode_like_count"
            ),
            "reason": quality_result["reason"]
        }

    # 3. Значения:
    # "Без товарного знака", "нет", "отсутствует", "не указан"
    if normalized_brand in no_brand_values:
        detected_brands = find_trusted_brands(text)

        if detected_brands:
            return {
                "status": "REVIEW",
                "declared_brand": brand,
                "detected_brands": detected_brands,
                "method": "no_brand_conflict_review",
                "reason": (
                    "Указано отсутствие товарного знака, "
                    "но в наименовании найдено упоминание бренда"
                )
            }

        return {
            "status": "REVIEW",
            "declared_brand": brand,
            "method": "no_brand_review",
            "reason": (
                "Указано отсутствие товарного знака. "
                "Недостаточно данных для автоматического подтверждения"
            )
        }

    # 4. Проверка формата значения товарного знака
    brand_quality_result = check_brand_input_quality(brand)

    if not brand_quality_result["is_valid"]:
        return {
            "status": "REVIEW",
            "declared_brand": brand,
            "method": "invalid_brand_format_review",
            "reason": brand_quality_result["reason"]
        }

    # 5. Точное вхождение бренда в наименование
    if normalized_brand in normalized_text:
        return {
            "status": "MATCH",
            "declared_brand": brand,
            "similarity": 100,
            "method": "exact_match",
            "reason": "Товарный знак найден в наименовании товара"
        }

    # 6. Проверенные вручную связи:
    # название аромата / линейки -> реальный бренд
    verified_relation = find_verified_product_relation(text)

    if verified_relation is not None:
        declared_canonical = normalize_for_matching(
            canonicalize_brand(brand)
        )

        verified_canonical = normalize_for_matching(
            canonicalize_brand(
                verified_relation["brand"]
            )
        )

        # Связь подтверждает заявленный бренд
        if declared_canonical == verified_canonical:
            return {
                "status": "MATCH",
                "declared_brand": brand,
                "associated_term": verified_relation["term"],
                "associated_brand": verified_relation["brand"],
                "method": "verified_relation_match",
                "reason": (
                    "В наименовании найдено название аромата "
                    "или линейки, связанное с заявленным товарным знаком"
                )
            }

        # Связь указывает на другой бренд
        return {
            "status": "MISMATCH",
            "declared_brand": brand,
            "detected_brands": [
                verified_relation["brand"]
            ],
            "associated_term": verified_relation["term"],
            "method": "verified_relation_conflict",
            "reason": (
                "В наименовании найдено название аромата "
                "или линейки, связанное с другим товарным знаком"
            )
        }

    # 7. Автоматически найденные устойчивые ассоциации
    associations = find_associated_brands(text)

    declared_canonical = normalize_for_matching(
        canonicalize_brand(brand)
    )

    for association in associations:
        associated_canonical = normalize_for_matching(
            canonicalize_brand(
                association["brand"]
            )
        )

        # Ассоциация подтверждает заявленный бренд
        if declared_canonical == associated_canonical:
            return {
                "status": "MATCH",
                "declared_brand": brand,
                "associated_term": association["term"],
                "associated_brand": association["brand"],
                "method": "association_match",
                "reason": (
                    "В наименовании найдено название, "
                    "устойчиво связанное с заявленным товарным знаком"
                )
            }

    # 8. Проверка прямого упоминания другого бренда
    direct_conflict = find_direct_brand_conflict(
        brand,
        text
    )

    if direct_conflict is not None:
        return {
            "status": "MISMATCH",
            "declared_brand": brand,
            "detected_brands": [
                direct_conflict["brand"]
            ],
            "associated_term": direct_conflict["term"],
            "method": "direct_brand_conflict",
            "reason": (
                "В наименовании явно обнаружен другой "
                "товарный знак"
            )
        }

    # 9. Fuzzy matching
    similarity = fuzz.partial_ratio(
        normalized_brand,
        normalized_text
    )

    if similarity >= 90:
        return {
            "status": "LIKELY_MATCH",
            "declared_brand": brand,
            "similarity": similarity,
            "method": "fuzzy_match",
            "reason": (
                "Обнаружено высокое сходство "
                "товарного знака и наименования"
            )
        }

    # 10. Поиск другого доверенного бренда
    conflict_status, conflicts = detect_trusted_mismatch(
        brand,
        text
    )

    if conflict_status == "MISMATCH":
        return {
            "status": "MISMATCH",
            "declared_brand": brand,
            "detected_brands": conflicts,
            "similarity": similarity,
            "method": "trusted_brand_conflict",
            "reason": (
                "В наименовании обнаружен другой "
                "доверенный товарный знак"
            )
        }

    # 11. Недостаточно информации
    return {
        "status": "REVIEW",
        "declared_brand": brand,
        "similarity": similarity,
        "method": "insufficient_evidence",
        "reason": (
            "Недостаточно данных для автоматического решения"
        )
    }

### Smoke-тесты финальной функции

`Smoke tests` — короткие контрольные проверки ключевых веток алгоритма:

- точное совпадение;
- совпадение по ассоциации;
- нечёткое совпадение;
- явный конфликт;
- отсутствие товарного знака;
- прямое упоминание другого бренда.

In [22]:
tests = [
    ("PARFUMS DE MARLY", "Parfums de Marly Delina Exclusif"),
    ("Byredo", "Парфюмерная вода Gypsy Water"),
    ("Franck Oliver", "FRANCK OLIVIER IN PINK"),
    ("Dior", "Police The Legendary Scent"),
    ("НЕ УКАЗАН", "PRADA AMBER POUR HOMME"),
    ("Mr.Tooth", "M.MICALLEF парфюмерная вода ДЕСИРТОКСИК 100ml"),
]

for brand, text in tests:
    result = check_trademark(brand, text)
    print(f"{brand!r} -> {result['status']} / {result.get('method')}")

print("\nРусская расшифровка smoke-тестов:")
for brand, text in tests:
    result = check_trademark(brand, text)
    print(
        f"{brand!r} -> "
        f"{status_ru(result['status'])} / "
        f"{method_ru(result.get('method'))}"
    )


'PARFUMS DE MARLY' -> MATCH / exact_match
'Byredo' -> MATCH / association_match
'Franck Oliver' -> MISMATCH / direct_brand_conflict
'Dior' -> MISMATCH / direct_brand_conflict
'НЕ УКАЗАН' -> REVIEW / no_brand_conflict_review
'Mr.Tooth' -> MISMATCH / direct_brand_conflict

Русская расшифровка smoke-тестов:
'PARFUMS DE MARLY' -> MATCH (соответствие) / точное совпадение
'Byredo' -> MATCH (соответствие) / статистическая ассоциация подтверждает бренд
'Franck Oliver' -> MISMATCH (несоответствие) / прямое упоминание другого бренда
'Dior' -> MISMATCH (несоответствие) / прямое упоминание другого бренда
'НЕ УКАЗАН' -> REVIEW (ручная проверка) / бренд не указан, но в тексте найден бренд
'Mr.Tooth' -> MISMATCH (несоответствие) / прямое упоминание другого бренда


## 7. Регрессионная проверка на 5000 случайных строк

`Regression check` (**регрессионная проверка**) показывает, не сломали ли новые изменения ранее работающие сценарии.

Используется воспроизводимая выборка `random_state=42`.

Важно: распределение статусов на 5000 строках показывает поведение алгоритма, но **не является метрикой точности**.

In [23]:
random_df = data[["brand", "text", "brand_in_text"]].sample(
    n=5000,
    random_state=42
).copy()

random_df["check_result"] = random_df.apply(
    lambda row: check_trademark(row["brand"], row["text"]),
    axis=1
)
random_df["final_status"] = random_df["check_result"].apply(lambda x: x["status"])
random_df["final_method"] = random_df["check_result"].apply(lambda x: x.get("method"))

print(random_df["final_status"].value_counts())
print()
print(random_df["final_method"].value_counts())

random_df["final_status_ru"] = random_df["final_status"].apply(status_ru)
random_df["final_method_ru"] = random_df["final_method"].apply(method_ru)

status_counts = random_df["final_status"].value_counts()
method_counts = random_df["final_method"].value_counts()

print("\nСтатусы с русскими пояснениями:")
display(pd.DataFrame({
    "Статус": status_counts.index,
    "Русское пояснение": [status_ru(x) for x in status_counts.index],
    "Количество": status_counts.values,
}))

print("\nМетоды с русскими пояснениями:")
display(pd.DataFrame({
    "Метод": method_counts.index,
    "Русское пояснение": [method_ru(x) for x in method_counts.index],
    "Количество": method_counts.values,
}))


final_status
MATCH           2636
REVIEW          2251
MISMATCH          93
LIKELY_MATCH      20
Name: count, dtype: int64

final_method
exact_match                    2454
insufficient_evidence          1361
no_brand_review                 761
association_match               175
invalid_brand_format_review     117
direct_brand_conflict            93
fuzzy_match                      20
no_brand_conflict_review          9
verified_relation_match           7
missing_data                      2
input_quality_review              1
Name: count, dtype: int64

Статусы с русскими пояснениями:


,Статус,Русское пояснение,Количество
0,MATCH,MATCH (соответствие),2636
1,REVIEW,REVIEW (ручная проверка),2251
2,MISMATCH,MISMATCH (несоответствие),93
3,LIKELY_MATCH,LIKELY_MATCH (вероятное соответствие),20



Методы с русскими пояснениями:


,Метод,Русское пояснение,Количество
0,exact_match,точное совпадение,2454
1,insufficient_evidence,недостаточно данных для автоматического решения,1361
2,no_brand_review,бренд не указан — ручная проверка,761
3,association_match,статистическая ассоциация подтверждает бренд,175
4,invalid_brand_format_review,некорректный формат товарного знака,117
5,direct_brand_conflict,прямое упоминание другого бренда,93
6,fuzzy_match,нечёткое совпадение,20
7,no_brand_conflict_review,"бренд не указан, но в тексте найден бренд",9
8,verified_relation_match,проверенная связь аромат/линейка → бренд,7
9,missing_data,отсутствуют необходимые данные,2


### Зафиксированный результат последнего прогона на 5000

Финальная версия дала:

- **2636 `MATCH` — соответствие**;
- **2251 `REVIEW` — ручная проверка**;
- **93 `MISMATCH` — несоответствие**;
- **20 `LIKELY_MATCH` — вероятное соответствие**.

Это распределение решений, а не оценка качества.

In [24]:
# Зафиксированный результат последнего прогона на 5000 строках
final_5000_status = pd.Series({
    "MATCH": 2636,
    "REVIEW": 2251,
    "MISMATCH": 93,
    "LIKELY_MATCH": 20,
}, name="count")

final_5000_methods = pd.Series({
    "exact_match": 2454,
    "insufficient_evidence": 1361,
    "no_brand_review": 761,
    "association_match": 175,
    "invalid_brand_format_review": 117,
    "direct_brand_conflict": 93,
    "fuzzy_match": 20,
    "no_brand_conflict_review": 9,
    "verified_relation_match": 7,
    "missing_data": 2,
    "input_quality_review": 1,
}, name="count")

print(final_5000_status)
print()
print(final_5000_methods)

print("\nЗафиксированные статусы с русскими пояснениями:")
display(pd.DataFrame({
    "Статус": final_5000_status.index,
    "Русское пояснение": [status_ru(x) for x in final_5000_status.index],
    "Количество": final_5000_status.values,
}))

print("\nЗафиксированные методы с русскими пояснениями:")
display(pd.DataFrame({
    "Метод": final_5000_methods.index,
    "Русское пояснение": [method_ru(x) for x in final_5000_methods.index],
    "Количество": final_5000_methods.values,
}))


MATCH           2636
REVIEW          2251
MISMATCH          93
LIKELY_MATCH      20
Name: count, dtype: int64

exact_match                    2454
insufficient_evidence          1361
no_brand_review                 761
association_match               175
invalid_brand_format_review     117
direct_brand_conflict            93
fuzzy_match                      20
no_brand_conflict_review          9
verified_relation_match           7
missing_data                      2
input_quality_review              1
Name: count, dtype: int64

Зафиксированные статусы с русскими пояснениями:


,Статус,Русское пояснение,Количество
0,MATCH,MATCH (соответствие),2636
1,REVIEW,REVIEW (ручная проверка),2251
2,MISMATCH,MISMATCH (несоответствие),93
3,LIKELY_MATCH,LIKELY_MATCH (вероятное соответствие),20



Зафиксированные методы с русскими пояснениями:


,Метод,Русское пояснение,Количество
0,exact_match,точное совпадение,2454
1,insufficient_evidence,недостаточно данных для автоматического решения,1361
2,no_brand_review,бренд не указан — ручная проверка,761
3,association_match,статистическая ассоциация подтверждает бренд,175
4,invalid_brand_format_review,некорректный формат товарного знака,117
5,direct_brand_conflict,прямое упоминание другого бренда,93
6,fuzzy_match,нечёткое совпадение,20
7,no_brand_conflict_review,"бренд не указан, но в тексте найден бренд",9
8,verified_relation_match,проверенная связь аромат/линейка → бренд,7
9,missing_data,отсутствуют необходимые данные,2


## 8. Ручная проверка и итоговая оценка baseline

После этапа разработки была создана отдельная ручная тестовая выборка из **100 товаров**.

Ручные метки:

- `MATCH` — соответствие;
- `MISMATCH` — несоответствие;
- `UNCERTAIN` — недостаточно данных для уверенного ответа.

13 `UNCERTAIN` исключены из основных бинарных метрик.

Важно: `association_map` строился по исходному датасету до строгого `train/test split` (**разделения на обучающую и тестовую части**), поэтому метрики следует трактовать как оценку текущего прототипа, а не как окончательную `production`-оценку.

In [25]:
# Итоговая ручная оценка: 100 товаров, из них 13 UNCERTAIN.
manual_label_distribution = pd.Series({
    "MATCH": 68,
    "MISMATCH": 19,
    "UNCERTAIN": 13,
}, name="count")

final_confusion = pd.DataFrame(
    {
        "MATCH": [50, 0],
        "MISMATCH": [1, 3],
        "REVIEW": [17, 16],
    },
    index=pd.Index(["MATCH", "MISMATCH"], name="manual_label")
)
final_confusion.columns.name = "predicted_label"

# Восстанавливаем y_true/y_pred из подтверждённой матрицы, чтобы пересчитать метрики.
y_true = (
    ["MATCH"] * 68
    + ["MISMATCH"] * 19
)
y_pred = (
    ["MATCH"] * 50
    + ["MISMATCH"] * 1
    + ["REVIEW"] * 17
    + ["MISMATCH"] * 3
    + ["REVIEW"] * 16
)

precision, recall, f1, support = precision_recall_fscore_support(
    y_true,
    y_pred,
    labels=["MATCH", "MISMATCH"],
    zero_division=0
)

metrics_df = pd.DataFrame({
    "class": ["MATCH", "MISMATCH"],
    "precision": precision,
    "recall": recall,
    "f1": f1,
    "support": support,
})

automatic_total = 50 + 1 + 3
automatic_correct = 50 + 3
coverage = automatic_total / 87
automatic_accuracy = automatic_correct / automatic_total

print("Распределение ручных меток:")
print(manual_label_distribution)
print("\nМатрица итоговой оценки:")
display(final_confusion)
print(f"Coverage: {coverage:.2%}")
print(f"Accuracy среди автоматических решений: {automatic_accuracy:.2%}")
display(metrics_df)

print("\nРусскоязычное представление итоговой оценки:")

display(pd.DataFrame({
    "Ручная метка": manual_label_distribution.index,
    "Русское пояснение": [status_ru(x) for x in manual_label_distribution.index],
    "Количество": manual_label_distribution.values,
}))

final_confusion_ru = final_confusion.rename(
    index={
        "MATCH": "MATCH (реальное соответствие)",
        "MISMATCH": "MISMATCH (реальное несоответствие)",
    },
    columns={
        "MATCH": "MATCH (алгоритм: соответствие)",
        "MISMATCH": "MISMATCH (алгоритм: несоответствие)",
        "REVIEW": "REVIEW (алгоритм: ручная проверка)",
    },
)

display(final_confusion_ru)

metrics_df_ru = metrics_df.rename(columns={
    "class": "Класс",
    "precision": "Precision (точность срабатываний)",
    "recall": "Recall (полнота)",
    "f1": "F1-score (F1-мера)",
    "support": "Support (число примеров)",
})

metrics_df_ru["Класс"] = metrics_df_ru["Класс"].map({
    "MATCH": "MATCH (соответствие)",
    "MISMATCH": "MISMATCH (несоответствие)",
})

print(f"Coverage (покрытие): {coverage:.2%}")
print(
    "Accuracy (доля правильных автоматических решений): "
    f"{automatic_accuracy:.2%}"
)
display(metrics_df_ru)


Распределение ручных меток:
MATCH        68
MISMATCH     19
UNCERTAIN    13
Name: count, dtype: int64

Матрица итоговой оценки:


predicted_label,MATCH,MISMATCH,REVIEW
manual_label,,,
MATCH,50,1,17
MISMATCH,0,3,16


Coverage: 62.07%
Accuracy среди автоматических решений: 98.15%


,class,precision,recall,f1,support
0,MATCH,1.00,0.735294,0.847458,68
1,MISMATCH,0.75,0.157895,0.260870,19



Русскоязычное представление итоговой оценки:


,Ручная метка,Русское пояснение,Количество
0,MATCH,MATCH (соответствие),68
1,MISMATCH,MISMATCH (несоответствие),19
2,UNCERTAIN,UNCERTAIN (неопределённый случай),13


predicted_label,MATCH (алгоритм: соответствие),MISMATCH (алгоритм: несоответствие),REVIEW (алгоритм: ручная проверка)
manual_label,,,
MATCH (реальное соответствие),50,1,17
MISMATCH (реальное несоответствие),0,3,16


Coverage (покрытие): 62.07%
Accuracy (доля правильных автоматических решений): 98.15%


,Класс,Precision (точность срабатываний),Recall (полнота),F1-score (F1-мера),Support (число примеров)
0,MATCH (соответствие),1.00,0.735294,0.847458,68
1,MISMATCH (несоответствие),0.75,0.157895,0.260870,19


## 9. Вывод

Итог алгоритмического `baseline` (**базового решения**):

- **Coverage (покрытие): 62,07%**
- **Accuracy (доля правильных решений среди автоматически принятых): 98,15%**

### `MATCH` — соответствие
- `Precision` (**точность срабатываний**) = **100%**
- `Recall` (**полнота**) = **73,53%**
- `F1-score` (**F1-мера**) = **84,75%**

### `MISMATCH` — несоответствие
- `Precision` (**точность срабатываний**) = **75%**
- `Recall` (**полнота**) = **15,79%**
- `F1-score` (**F1-мера**) = **26,09%**

Главное ограничение — низкий `Recall` класса `MISMATCH`: **16 из 19 реальных конфликтов ушли в `REVIEW`**.

Поэтому правила стоит сохранить как первый безопасный слой, а ML использовать для сложных `REVIEW`-случаев.

## 10. Следующий этап: Machine Learning

План:

1. строгий `train/test split` — обучающая и тестовая выборки;
2. построение статистических справочников только по `train`;
3. подготовка пар `brand + text → MATCH/MISMATCH`;
4. простой NLP-baseline: `TF-IDF + Logistic Regression` (**TF-IDF + логистическая регрессия**);
5. сравнение с алгоритмическим baseline;
6. при необходимости более сложная модель.

Цель ML — уменьшить долю `REVIEW` и повысить `Recall` класса `MISMATCH`, не потеряв высокую `Precision`.

## Справочные материалы

- RapidFuzz `partial_ratio` — нечёткое сравнение строк: https://rapidfuzz.github.io/RapidFuzz/Usage/fuzz.html
- scikit-learn Precision/Recall — точность срабатываний и полнота: https://scikit-learn.org/stable/auto_examples/model_selection/plot_precision_recall.html
- scikit-learn `classification_report` — Precision / Recall / F1: https://scikit-learn.org/stable/modules/generated/sklearn.metrics.classification_report.html
- Jupyter Notebook — код, Markdown и результаты в одном документе: https://jupyter-notebook.readthedocs.io/en/latest/notebook.html